In [ ]:
import pandas as pd
import requests


def fetch_observations(series_id, fred_api_key, start_date=None):
    params = {
        "series_id": series_id,
        "api_key": fred_api_key,
        "file_type": "json",
    }
    if start_date:
        params["observation_start"] = start_date

    response = requests.get("https://api.stlouisfed.org/fred/series/observations", params=params)
    response.raise_for_status()
    observations = response.json()["observations"]

    df = pd.DataFrame(observations)[["date", "value"]]
    df = df[df["value"] != "."]
    df["value"] = df["value"].astype(float)
    return df.rename(columns={"value": series_id})


def ingest_data(series, start_date=None):
    fred_api_key = dbutils.secrets.get(scope="finsight", key="fred-api-key")

    series_frames = [fetch_observations(series_id, fred_api_key, start_date) for series_id in series]

    indicators_pd = series_frames[0]
    for series_df in series_frames[1:]:
        indicators_pd = indicators_pd.merge(series_df, on="date", how="outer")
    indicators_pd["date"] = pd.to_datetime(indicators_pd["date"]).dt.date

    return spark.createDataFrame(indicators_pd)

In [ ]:
def update_config(data, params):
    max_date = data.select("date").agg({"date": "max"}).first()[0]
    return {**params, "start_date": str(max_date)}